In [10]:
import pandas as pd

In [11]:
discounted_rates = pd.read_csv('input/23062025_freight_rates_operating_multi.csv')

In [12]:
# Constants for rate adjustment
XGS_RATE_DISCOUNT = 0.06
XGS_FUEL_SURCHARGE = 0.3
XGS_LTL_REBATE = 0.1
STARNET_REBATE = 0.025


# Adjustment function
def adjust_rate(rate):
    inflation_rate = rate / (1 + XGS_RATE_DISCOUNT)
    fsc_rate = inflation_rate * (1 + XGS_FUEL_SURCHARGE)
    xgs_rebate = inflation_rate * XGS_LTL_REBATE
    star_net_rebate = (inflation_rate - xgs_rebate) * STARNET_REBATE
    final_rate = fsc_rate - xgs_rebate - star_net_rebate
    return final_rate

In [13]:
def scale_freight_rates_by_condition(df, commodity, unit, freight_class_cols, multiplier):
    """
    Scales freight rate columns by a given multiplier for rows matching a specific commodity and unit.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing freight rate columns.
    - commodity (str): The value in 'commodity_group' to match (e.g. "1VNL").
    - unit (str): The value in 'unit' column to match (e.g. "CWT").
    - freight_class_cols (list): List of freight class columns to scale.
    - multiplier (float): Value to multiply the freight class columns by.

    Returns:
    - pd.DataFrame: The modified DataFrame with rates scaled accordingly.
    """
    df = df.copy()
    mask = (df["commodity_group"] == commodity) & (df["unit"] == unit)
    df.loc[mask, freight_class_cols] = df.loc[mask, freight_class_cols] * multiplier
    print(f"✅ Scaled {commodity} with unit {unit} by {multiplier} for freight classes.")
    return df

In [14]:
# List of freight class columns
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']
# Apply adjustments to each freight class column
for col in freight_classes:
    if col in discounted_rates.columns:
        discounted_rates[col] = adjust_rate(pd.to_numeric(discounted_rates[col], errors='coerce'))

print("✅ Vendor rates adjusted for FSC, XGS rebate, and StarNet rebate.")

✅ Vendor rates adjusted for FSC, XGS rebate, and StarNet rebate.


In [15]:
# Step 6: Normalize Vendor Rates from $/CWT to $/LBS

# Identify rows where unit is CWT (used for 1VNL)
vendor_cwt_mask = (discounted_rates["commodity_group"] == "1VNL") & (discounted_rates["unit"] == "CWT")

# List of freight class columns to scale
freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Convert vendor rates from $/CWT to $/LBS
discounted_rates.loc[vendor_cwt_mask, freight_class_cols] = discounted_rates.loc[vendor_cwt_mask, freight_class_cols] / 100

print("✅ Converted vendor CWT rates to $/LBS for comparability.")

✅ Converted vendor CWT rates to $/LBS for comparability.


In [16]:
# convert the rates from $/LBS to $/SQFT
freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

discounted_rates = scale_freight_rates_by_condition(
    df=discounted_rates,
    commodity="1VNL",
    unit="CWT",
    freight_class_cols=freight_class_cols,
    multiplier=1.2
)


✅ Scaled 1VNL with unit CWT by 1.2 for freight classes.


In [18]:
discounted_rates.to_excel('updated_freight_rates2.xlsx', index=False)